In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate
# 必须添加的导入语句
import random
import numpy as np
import torch

# 如果使用GPU需要额外导入
from torch.backends import cudnn

E:\Anaconda\envs\yl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码。
设置 os.environ['HF_ENDPOINT'] = \'https://hf-mirror.com' ，这样做具体改变了什么？
为什么要设置HF_ENDPOINT=\'https://hf-mirror.com'而非直接使用官方源？
dataset = datasets.load_dataset("bentrevett/multi30k") 这行代码具体完成了什么操作？

1.这会将Hugging Face库的默认API端点从官方源 https://huggingface.co 改为镜像站 https://hf-mirror.com
2.原因：在中国大陆访问官方源可能速度较慢或不稳定、镜像站通常部署在国内服务器，下载速度更快、避免因网络问题导致长时间等待或中断
3.数据集定位、自动下载、数据处理、返回一个 DatasetDict 对象

In [5]:
os.environ['HF_ENDPOINT'] = 'http://hf-mirror.com'
dataset = datasets.load_dataset("bentrevett/multi30k")

Using the latest cached version of the dataset since bentrevett/multi30k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\long\.cache\huggingface\datasets\bentrevett___multi30k\default\0.0.0\4589883f3d09d4ef6361784e03f0ead219836469 (last modified on Tue May 27 12:20:13 2025).


## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [9]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [11]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [13]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [15]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


是一个用于处理双鱼平行语料（英语-德语）的文本预处理函数

In [17]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


表示句子的开始，帮助模型知道合适开始生成序列。
表示句子的结束，高数模型何时停止生成。
这里的map 是Hugging Face datasets库提供的方法， 用于批量处理数据集中的每个样本

In [21]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 9131.03 examples/s]


## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [23]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [25]:
from torchtext.vocab import build_vocab_from_iterator
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

E:\Anaconda\envs\yl\lib\site-packages\torchtext\vocab\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
E:\Anaconda\envs\yl\lib\site-packages\torchtext\utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [27]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [29]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [31]:
en_vocab["the"]

7

In [33]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [35]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [37]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [39]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

词汇表中没有 “crime”这个词

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


将分词后的文本tokens转换为数值ID

In [41]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [43]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 10482.27 examples/s]


## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [45]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [47]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [49]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

1.生成一个自定义的collate_fn函数，用于将一批样本整理成模型可处理的格式（填充短序列到相同长度）
2.创建一个DataLoader，用于批量加载数据集并自动调用上述collate_fn处理数据。

In [51]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [53]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [55]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。


以下是对这段PyTorch代码的详细解释，涵盖输入参数、核心组件、`forward`函数处理流程及输出：


### **一、输入参数**
```python
def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
```
- **`input_dim`**：输入词汇表的大小（即词嵌入的索引范围，通常为词典大小）。  
- **`embedding_dim`**：词嵌入向量的维度（每个单词映射到的稠密向量长度）。  
- **`hidden_dim`**：LSTM隐藏层的维度（每个LSTM单元的输出维度）。  
- **`n_layers`**：LSTM的层数（堆叠的LSTM层数量，默认单向）。  
- **`dropout`**：Dropout概率（用于防止过拟合，丢弃神经元的概率）。


### **二、核心组件**
#### 1. **词嵌入层（Embedding Layer）**
```python
self.embedding = nn.Embedding(input_dim, embedding_dim)
```
- **作用**：将输入的离散词索引（如`src`中的整数）映射为连续的稠密向量（词嵌入）。  
- **输入**：形状为`[src length, batch size]`的长整型张量（序列长度×批量大小）。  
- **输出**：形状为`[src length, batch size, embedding_dim]`的浮点型张量。

#### 2. **LSTM层（LSTM Layer）**
```python
self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
```
- **作用**：对嵌入后的序列进行特征提取，捕捉上下文依赖关系。  
- **关键参数**：  
  - `embedding_dim`：输入维度（即词嵌入向量的维度）。  
  - `hidden_dim`：隐藏层维度（LSTM单元的输出维度）。  
  - `n_layers`：LSTM层数，每层的输出作为下一层的输入。  
  - `dropout=dropout`：在层间连接中应用Dropout（仅当`n_layers > 1`时有效）。  
- **默认配置**：单向LSTM（`bidirectional=False`），因此输出不包含双向拼接。

#### 3. **Dropout层（Dropout Layer）**
```python
self.dropout = nn.Dropout(dropout)
```
- **作用**：在词嵌入层后应用Dropout，随机丢弃部分神经元以缓解过拟合。  
- **应用位置**：仅在`forward`函数中对词嵌入结果进行Dropout，LSTM层内部的Dropout由`nn.LSTM`的`dropout`参数控制。


### **三、forward函数处理流程**
```python
def forward(self, src):
    # src = [src length, batch size]
    embedded = self.dropout(self.embedding(src))
    # embedded = [src length, batch size, embedding dim]
    outputs, (hidden, cell) = self.rnn(embedded)
    # outputs = [src length, batch size, hidden dim * n directions]
    # hidden = [n layers * n directions, batch size, hidden dim]
    # cell = [n layers * n directions, batch size, hidden dim]
    return hidden, cell
```

#### 1. **输入处理**
- **输入`src`**：形状为`[src length, batch size]`的长整型张量，表示输入序列的词索引。  
  - 例如：若输入序列长度为10，批量大小为32，则`src`形状为`(10, 32)`。

#### 2. **词嵌入与Dropout**
```python
embedded = self.dropout(self.embedding(src))
```
- **步骤**：  
  1. 通过`self.embedding(src)`将词索引转换为词嵌入向量，输出形状为`(src length, batch size, embedding_dim)`。  
  2. 通过`self.dropout`对嵌入向量应用Dropout，随机归零部分元素以防止过拟合。

#### 3. **LSTM层计算**
```python
outputs, (hidden, cell) = self.rnn(embedded)
```
- **输入`embedded`**：形状为`(src length, batch size, embedding_dim)`的张量，作为LSTM的输入序列。  
- **LSTM输出**：  
  - **`outputs`**：LSTM所有时间步的隐藏状态，形状为`(src length, batch size, hidden_dim)`。  
    - 由于是单向LSTM，`n directions=1`，因此无需拼接双向结果。  
  - **`hidden`**：最后一个时间步的隐藏状态，形状为`(n_layers, batch size, hidden_dim)`。  
  - **`cell`**：最后一个时间步的细胞状态，形状与`hidden`相同。  
- **注意**：代码中仅返回`hidden`和`cell`，未使用`outputs`（可能用于解码器的初始状态）。


### **四、输出**
```python
return hidden, cell
```
- **输出内容**：  
  - **`hidden`**：LSTM最后一层、最后一个时间步的隐藏状态，形状为`(n_layers, batch size, hidden_dim)`。  
  - **`cell`**：LSTM最后一层、最后一个时间步的细胞状态，形状与`hidden`相同。  
- **用途**：通常作为解码器（如Seq2Seq模型中的Decoder）的初始隐藏状态和细胞状态，用于初始化解码过程。


### **五、总结**
- **功能**：该编码器将输入序列转换为词嵌入，通过LSTM提取序列特征，并输出最终的隐藏状态和细胞状态，用于后续任务（如机器翻译、文本生成等）。  
- **关键特点**：  
  - 使用Dropout对词嵌入层进行正则化。  
  - 支持多层LSTM堆叠，捕捉更深层次的语义信息。  
  - 输出可直接用于初始化解码器，实现序列到序列的建模。这段代码定义了一个基于LSTM的编码器（Encoder）类，是序列到序列（Seq2Seq）模型的关键组件，常用于机器翻译等任务。以下是对代码的详细解释：

### 输入参数
- `input_dim`: 输入词汇表的大小，决定了词嵌入层的权重矩阵维度。
- `embedding_dim`: 词向量的维度，用于将输入的离散词索引转换为连续的向量表示。
- `hidden_dim`: LSTM隐藏状态的维度，影响模型的表示能力。
- `n_layers`: LSTM的层数，深层结构可学习更复杂的序列模式。
- `dropout`: Dropout比率，用于防止过拟合，仅在多层LSTM的非顶层应用。

### 核心组件
1. **词嵌入层 (`nn.Embedding`)**
   - 功能：将输入的词索引（整数）映射为对应的词向量。
   - 示例：若`input_dim=1000`，`embedding_dim=256`，则每个词被表示为256维向量。

2. **LSTM层 (`nn.LSTM`)**
   - 结构：多层双向LSTM，处理序列数据时能捕获长距离依赖关系。
   - 输入：词嵌入向量序列。
   - 输出：
     - `outputs`: 所有时间步的隐藏状态。
     - `hidden`: 最后时间步的隐藏状态。
     - `cell`: 最后时间步的细胞状态（LSTM特有）。

3. **Dropout层 (`nn.Dropout`)**
   - 功能：训练时随机丢弃部分神经元，减少神经元间的共适应，提升泛化能力。

### `forward`函数处理流程
1. **输入处理**
   - 输入`src`形状：`[序列长度, 批量大小]`，如`[10, 32]`表示32个样本，每个样本10个词。

2. **词嵌入与Dropout**
   ```python
   embedded = self.dropout(self.embedding(src))
   ```
   - 嵌入后形状：`[序列长度, 批量大小, 嵌入维度]`。
   - Dropout在嵌入层后应用，增强模型鲁棒性。

3. **LSTM处理**
   ```python
   outputs, (hidden, cell) = self.rnn(embedded)
   ```
   - `outputs`包含所有时间步的隐藏状态，形状为`[序列长度, 批量大小, 隐藏维度×方向数]`。
   - `hidden`和`cell`为最终时间步的状态，形状为`[层数×方向数, 批量大小, 隐藏维度]`。

4. **输出**
   - 返回`hidden`和`cell`，作为解码器的初始状态，传递序列的语义信息。

### 输出说明
- `hidden`: 编码器最终隐藏状态，捕获整个输入序列的语义摘要。
- `cell`: 编码器最终细胞状态，保存长期记忆信息。
- 这两个状态将作为解码器的初始状态，指导目标序列的生成。

### 应用场景
该编码器常用于神经机器翻译，如将英语句子编码为语义向量，再由解码器转换为中文。其核心作用是将输入序列压缩为固定维度的语义表示，供后续生成任务使用。

In [57]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

### Decoder工作流程详解

这个Decoder类是序列到序列模型的核心组件，负责将编码器输出的语义表示转换为目标语言序列。以下是其工作流程的详细描述：


### **一、初始化参数**
```python
def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
```
- **`output_dim`**：输出词汇表的大小（如目标语言的词典大小）。  
- **`embedding_dim`**：词嵌入向量的维度。  
- **`hidden_dim`**：LSTM隐藏层的维度（需与Encoder匹配）。  
- **`n_layers`**：LSTM的层数（需与Encoder匹配）。  
- **`dropout`**：Dropout概率，防止过拟合。


### **二、核心组件**
#### 1. **词嵌入层（Embedding）**
```python
self.embedding = nn.Embedding(output_dim, embedding_dim)
```
- **作用**：将输入的目标词索引转换为连续的词向量。

#### 2. **LSTM层**
```python
self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
```
- **作用**：基于当前输入词和上一时间步的隐藏状态，生成新的隐藏状态。  
- **输入**：  
  - `embedded`：当前词的嵌入向量。  
  - `(hidden, cell)`：上一时间步的隐藏状态和细胞状态（初始来自Encoder）。

#### 3. **全连接层（输出层）**
```python
self.fc_out = nn.Linear(hidden_dim, output_dim)
```
- **作用**：将LSTM的输出映射到词汇表大小的维度，用于预测下一个词的概率分布。

#### 4. **Dropout层**
```python
self.dropout = nn.Dropout(dropout)
```
- **作用**：对词嵌入结果应用Dropout，防止过拟合。


### **三、forward函数处理流程**
#### 1. **输入处理**
```python
def forward(self, input, hidden, cell):
    # input = [batch size]
    # hidden = [n layers, batch size, hidden dim]
    # cell = [n layers, batch size, hidden dim]
    input = input.unsqueeze(0)
    # input = [1, batch size]
```
- **输入参数**：  
  - `input`：当前时间步的输入词索引（批量中的每个样本对应一个词）。  
  - `hidden`和`cell`：上一时间步的隐藏状态和细胞状态（初始来自Encoder）。  
- **处理**：  
  - 通过`unsqueeze(0)`为输入添加序列长度维度（Decoder每次处理一个词，序列长度为1）。

#### 2. **词嵌入与Dropout**
```python
embedded = self.dropout(self.embedding(input))
# embedded = [1, batch size, embedding dim]
```
- **步骤**：  
  1. 将输入词索引转换为词向量。  
  2. 对嵌入向量应用Dropout。

#### 3. **LSTM计算**
```python
output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
# output = [1, batch size, hidden dim]
# hidden = [n layers, batch size, hidden dim]
# cell = [n layers, batch size, hidden dim]
```
- **输入**：  
  - `embedded`：当前词的嵌入向量。  
  - `(hidden, cell)`：上一时间步的状态（初始为Encoder的最终状态）。  
- **输出**：  
  - `output`：当前时间步的隐藏状态。  
  - `hidden`和`cell`：更新后的隐藏状态和细胞状态，将用于下一时刻。

#### 4. **输出预测**
```python
prediction = self.fc_out(output.squeeze(0))
# prediction = [batch size, output dim]
```
- **步骤**：  
  1. 通过`squeeze(0)`移除序列长度维度（Decoder每次只处理一个词）。  
  2. 通过全连接层将LSTM输出映射到词汇表大小的维度。  
- **输出**：  
  - `prediction`：预测的下一个词的概率分布（未经过softmax）。


### **四、返回值**
```python
return prediction, hidden, cell
```
- **`prediction`**：预测的下一个词的概率分布，形状为`[batch size, output dim]`。  
- **`hidden`和`cell`**：更新后的隐藏状态和细胞状态，用于下一时刻的解码。


### **五、解码器在Seq2Seq模型中的应用**
1. **初始状态**：  
   - `hidden`和`cell`初始化为Encoder的最终状态，携带源序列的语义信息。

2. **自回归解码**：  
   - 在训练阶段，解码器使用真实标签（teacher forcing）作为输入。  
   - 在推理阶段，解码器使用自己的预测作为下一时刻的输入。

3. **流程示例**：  
   ```python
   # 推理阶段简化流程（假设已训练好模型）
   encoder_outputs, hidden, cell = encoder(src)  # 编码源序列
   input = target_vocab['<sos>']  # 初始输入为开始符号
   for i in range(max_len):
       prediction, hidden, cell = decoder(input, hidden, cell)
       next_token = prediction.argmax(1)  # 选择概率最高的词
       input = next_token  # 下一时刻的输入
   ```


### **六、总结**
- **功能**：解码器通过LSTM和全连接层，结合编码器的语义信息，逐词生成目标序列。  
- **特点**：  
  - 每次处理一个时间步，自回归地生成序列。  
  - 隐藏状态和细胞状态在时间步之间传递，累积上下文信息。  
  - 最终输出为词汇表上的概率分布，可通过softmax转换为概率。

class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

这段代码实现了一个完整的序列到序列（Seq2Seq）模型，它结合了之前的编码器和解码器，并引入了teacher forcing训练机制。以下是对代码的详细解释：


### **一、Seq2Seq模型整体架构**
```python
def __init__(self, encoder, decoder, device):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.device = device
    # 确保编码器和解码器的隐藏层维度和层数一致
    assert encoder.hidden_dim == decoder.hidden_dim, "Hidden dimensions must match!"
    assert encoder.n_layers == decoder.n_layers, "Number of layers must match!"
```
- **核心组件**：  
  - `encoder`：处理输入序列，提取语义信息。  
  - `decoder`：基于编码器的输出，生成目标序列。  
  - `device`：指定计算设备（CPU/GPU）。  
- **约束条件**：  
  - 编码器和解码器的隐藏层维度（`hidden_dim`）和层数（`n_layers`）必须相同，以确保状态传递的兼容性。


### **二、forward函数流程**
#### 1. **输入与初始化**
```python
def forward(self, src, trg, teacher_forcing_ratio):
    # src = [src_len, batch_size]
    # trg = [trg_len, batch_size]
    batch_size = trg.shape[1]
    trg_length = trg.shape[0]
    trg_vocab_size = self.decoder.output_dim
    
    # 初始化存储解码器输出的张量
    outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
```
- **输入参数**：  
  - `src`：源序列（如英语句子）。  
  - `trg`：目标序列（如中文翻译）。  
  - `teacher_forcing_ratio`：使用teacher forcing的概率（0~1之间）。  
- **输出张量**：  
  - `outputs`：存储解码器在每个时间步的预测结果。


#### 2. **编码阶段**
```python
hidden, cell = self.encoder(src)
```
- **过程**：  
  - 将源序列`src`输入编码器，得到最终的隐藏状态`hidden`和细胞状态`cell`。  
- **作用**：  
  - `hidden`和`cell`作为解码器的初始状态，携带源序列的语义信息。


#### 3. **解码阶段（带teacher forcing）**
```python
# 第一个输入为<sos>标记（目标序列的第一个token）
input = trg[0, :]  # [batch_size]

for t in range(1, trg_length):
    # 解码器处理当前输入，生成预测和新状态
    output, hidden, cell = self.decoder(input, hidden, cell)
    outputs[t] = output  # 保存当前时间步的预测
    
    # 决定是否使用teacher forcing
    teacher_force = random.random() < teacher_forcing_ratio
    top1 = output.argmax(1)  # 预测的下一个token
    
    # 下一个输入：如果使用teacher forcing，则用真实标签；否则用预测值
    input = trg[t] if teacher_force else top1
```
- **关键步骤**：  
  1. **初始输入**：解码器的第一个输入是目标序列的起始标记（`<sos>`）。  
  2. **循环解码**：  
     - 每次迭代，解码器基于当前输入和前一时间步的状态，预测下一个token。  
     - 使用`teacher_forcing_ratio`决定是否使用真实标签作为下一个输入。  
  3. **状态更新**：解码器的隐藏状态和细胞状态在每次迭代中更新，累积上下文信息。


#### 4. **返回结果**
```python
return outputs
```
- **输出**：  
  - `outputs`：形状为`[trg_len, batch_size, trg_vocab_size]`，包含解码器在每个时间步对目标词汇表的预测分布。


### **三、Teacher Forcing机制**
#### 1. **核心思想**
- **训练时**：以概率`teacher_forcing_ratio`使用真实标签（而非模型预测）作为下一个输入。  
- **目的**：加速训练收敛，避免早期错误累积（模型可能因早期预测错误而偏离正确路径）。

#### 2. **工作流程**
1. **随机决策**：  
   ```python
   teacher_force = random.random() < teacher_forcing_ratio
   ```
   - 若`teacher_force`为`True`，使用真实标签`trg[t]`作为下一个输入。  
   - 若为`False`，使用模型预测的最高概率token`top1`。

2. **动态调整**：  
   - 通常在训练初期使用较高的`teacher_forcing_ratio`（如0.75），后期逐渐降低。  
   - 极端情况：  
     - `ratio=1`：完全使用真实标签（纯teacher forcing）。  
     - `ratio=0`：完全使用模型预测（自回归解码，与推理阶段一致）。

#### 3. **优缺点**
- **优点**：  
  - 训练更稳定，尤其是长序列任务。  
  - 帮助模型学习到更准确的条件概率分布。  
- **缺点**：  
  - 可能导致“暴露偏差”（训练和推理时行为不一致）。  
  - 过高的比例可能使模型过度依赖真实标签，降低泛化能力。


### **四、总结**
#### 1. **Seq2Seq模型工作流程**
1. **编码**：将源序列压缩为语义向量（`hidden`和`cell`）。  
2. **解码**：从起始标记开始，结合语义向量和teacher forcing机制，逐词生成目标序列。  
3. **预测**：最终输出每个时间步的词汇表概率分布，可通过`argmax`获取预测词。

#### 2. **应用场景**
- 机器翻译、文本摘要、对话系统等序列生成任务。

#### 3. **改进方向**
- 引入注意力机制（Attention）增强长序列处理能力。  
- 动态调整teacher forcing比例（如随训练轮数衰减）。  
- 使用计划采样（Scheduled Sampling）平衡训练和推理行为。

In [61]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

# 模型训练

模型初始化

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [63]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)
#解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)
#Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

权重初始化

In [65]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [67]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


优化器 optimizer

In [69]:
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [71]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [73]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        src = batch["de_ids"].to(device)
        trg = batch["en_ids"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [75]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

# 模型训练

In [87]:
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [09:18<00:00, 558.38s/it]

	Train Loss:   5.028 | Train PPL: 152.561
	Valid Loss:   4.846 | Valid PPL: 127.272


# 模型验证

In [88]:
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

In [89]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [90]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [91]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [92]:
translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'a',
 'shirt',
 'is',
 'a',
 'a',
 'a',
 '.',
 '<eos>']